In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import KMeans
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [47]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean_tags_final.parquet")

In [48]:
model_df = pd.read_parquet(DATA_PATH)

In [49]:
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(model_df["tags_genre_mood"])

In [39]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)

In [50]:
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_reduced)
model_df["cluster"] = clusters

In [51]:
def get_top_tags_per_cluster(X, clusters, feature_names, top_n=10):
    cluster_tags = {}
    
    for cluster in np.unique(clusters):
        idx = clusters == cluster
        mean_vals = X[idx].mean(axis=0)
        top_idx = np.argsort(mean_vals)[-top_n:]
        cluster_tags[cluster] = [feature_names[i] for i in top_idx]
        
    return cluster_tags

top_tags = get_top_tags_per_cluster(X, clusters, mlb.classes_)

for k, v in top_tags.items():
    print(f"Cluster {k}: {v}")

Cluster 0: ['dance', 'downtempo', 'jazz', 'blues', 'alternative rnb', 'neo soul', 'hip hop', 'funk', 'rnb', 'soul']
Cluster 1: ['british invasion', 'heavy metal', 'art rock', 'progressive rock', 'psychedelic', 'blues', 'psychedelic rock', 'blues rock', 'hard rock', 'classic rock']
Cluster 2: ['classic rock', 'blues', 'country', 'lo fi', 'alt country', 'americana', 'folk rock', 'acoustic', 'indie folk', 'folk']
Cluster 3: ['score', 'composers', 'modern classical', 'ambient', 'piano', 'soundtrack', 'contemporary classical', 'composer', 'classical', 'instrumental']
Cluster 4: ['art pop', 'grunge', 'metal', 'garage rock', 'punk rock', 'psychedelic', 'hard rock', 'lo fi', 'punk', 'britpop']
Cluster 5: ['acid jazz', 'jazz', 'nu jazz', 'instrumental', 'idm', 'lounge', 'ambient', 'trip hop', 'chillout', 'downtempo']
Cluster 6: ['depeche mode', 'dance', 'lo fi', 'indietronica', 'electro', 'art pop', 'post punk', 'electropop', 'new wave', 'synthpop']
Cluster 7: ['garage rock', 'art rock', 'noise

In [34]:
cluster_names = {
    0: "Classical / Soundtrack",
    1: "UK Indie / Post-Punk",
    2: "Classic / Psychedelic Rock",
    3: "Art Pop / Indietronica",
    4: "Punk / Hardcore",
    5: "Experimental Ambient",
    6: "Folk / Americana",
    7: "Chill Electronic / Trip-Hop",
    8: "Mixed Groove / Alt",
    9: "Art Rock / Experimental Indie"
}

In [35]:
model_df["cluster_name"] = model_df["cluster"].map(cluster_names)
model_df["cluster_name"].value_counts()

cluster_name
Mixed Groove / Alt               15377
UK Indie / Post-Punk             14825
Art Pop / Indietronica            9409
Folk / Americana                  7896
Experimental Ambient              6832
Art Rock / Experimental Indie     4807
Classic / Psychedelic Rock        4728
Chill Electronic / Trip-Hop       3405
Punk / Hardcore                   3118
Classical / Soundtrack            2426
Name: count, dtype: int64

In [10]:
# why not audio features?
# Spotify audio features were unavailable due to API limitations (403 errors), so I used Last.fm tags as a semantic representation of music, which actually captures genre and mood more directly.